<a href="https://colab.research.google.com/github/gcalanch/DMA-Caras/blob/main/Eigenfaces_GCv2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# conexion al Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Cargar las librerias que vamos a utilizar

In [2]:
import cv2
import math
import numpy as np
import os
import pickle
import random
import sys
import numpy as np
from sklearn.manifold import Isomap
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.model_selection import train_test_split
#!pip install --upgrade numpy
#!pip install --upgrade scipy

# Preprocesamos imagenes
## La carpeta de origen se llama Eigenfaces


In [3]:
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

# Definicion de ruta de conexion - Origen - Destino imagenes

img_size=(60,60)

ruta_entrada = '/content/drive/MyDrive/DMA/Eigenfaces'
ruta_salida = f'/content/drive/MyDrive/DMA/Eigenfaces2-{img_size[0]}x{img_size[1]}'


# prompt: verificar si la carpeta en "ruta_salida" existe. Si no existe, crearla

if not os.path.exists(ruta_salida):
  os.makedirs(ruta_salida)

def procesar_archivos_en_carpetas(ruta_principal,ruta_final, imgsize):
  """Recorre las carpetas dentro de la ruta principal y procesa los archivos.

  Args:
    ruta_principal: La ruta de la carpeta principal.
  """

  fraccion = 8


  x_inicial = imgsize[0] // (fraccion - 2)
  y_inicial = imgsize[1] // (fraccion - 2)
  alto = imgsize[1]
  ancho = imgsize[0]
  multip = fraccion / (fraccion - 2)
  img_size = (math.floor(ancho * multip) , math.floor(alto * multip))


  print(f"Ruta principal: {ruta_principal}")
  for carpeta_actual, _, archivos in os.walk(ruta_principal):
    print(f"Carpeta actual: {carpeta_actual}")
    for archivo in archivos:
      print(f"Archivo: {archivo}")
      ruta_completa = os.path.join(carpeta_actual, archivo)
      try:

            img = cv2.imread(ruta_completa, cv2.IMREAD_GRAYSCALE)

            img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)

            if img is None:
                print(f"⚠️ No se pudo leer: {ruta_completa}")
                continue

            # Detectar rostros
            faces = face_cascade.detectMultiScale(img, scaleFactor=1.1, minNeighbors=10, minSize=(110, 110))

            for i, (x, y, w, h) in enumerate(faces):
                face = img[y:y+h, x:x+w]  # Recortar rostro
                face_resized = cv2.resize(face, img_size)  # Redimensionar

                face_cutted = face_resized[y_inicial:y_inicial+alto, x_inicial:x_inicial+ancho]

                # Crear carpeta de salida manteniendo la estructura original
                relative_path = os.path.relpath(carpeta_actual, ruta_entrada)
                output_folder = os.path.join(ruta_salida, relative_path)
                os.makedirs(output_folder, exist_ok=True)

                # Guardar rostro procesado
                output_path = os.path.join(output_folder, f"{os.path.splitext(archivo)[0]}_face{i}.jpg")
                cv2.imwrite(output_path, face_cutted)
                print(f"✅ Guardado: {output_path}")


      except Exception as e:
          print(f"❌ Error procesando {ruta_completa}: {e}")


      except Exception as e:
        print(f"Error al procesar el archivo {ruta_completa}: {e}")

In [4]:
# Preprocesamos imagenes (escala de grises, recortes y escalado)
print("Procesando archivos")
procesar_archivos_en_carpetas(ruta_entrada, ruta_salida, img_size)
print("Proceso inicial terminado ...")

Procesando archivos
Ruta principal: /content/drive/MyDrive/DMA/Eigenfaces
Carpeta actual: /content/drive/MyDrive/DMA/Eigenfaces
Carpeta actual: /content/drive/MyDrive/DMA/Eigenfaces/Federico
Archivo: 20250321_140311.jpg
✅ Guardado: /content/drive/MyDrive/DMA/Eigenfaces2-60x60/Federico/20250321_140311_face0.jpg
Archivo: IMG_1182.JPG
✅ Guardado: /content/drive/MyDrive/DMA/Eigenfaces2-60x60/Federico/IMG_1182_face0.jpg
Archivo: IMG_1179.JPG
Archivo: IMG_1078.JPG
✅ Guardado: /content/drive/MyDrive/DMA/Eigenfaces2-60x60/Federico/IMG_1078_face0.jpg
Archivo: IMG_1079.JPG
✅ Guardado: /content/drive/MyDrive/DMA/Eigenfaces2-60x60/Federico/IMG_1079_face0.jpg
Archivo: IMG_1186.JPG
✅ Guardado: /content/drive/MyDrive/DMA/Eigenfaces2-60x60/Federico/IMG_1186_face0.jpg
Archivo: 1742907850046.jpg
✅ Guardado: /content/drive/MyDrive/DMA/Eigenfaces2-60x60/Federico/1742907850046_face0.jpg
Archivo: 1742907849997.jpg
✅ Guardado: /content/drive/MyDrive/DMA/Eigenfaces2-60x60/Federico/1742907849997_face0.jpg
✅ Gu

## el preprocesado queda en la carpeta Eigenfaces2-60x60

### copie una version de esa carpeta a "origen" para usar a continuacion

In [11]:
# origen de datos preprocesados
ruta_origen = "/content/drive/MyDrive/DMA/Eigenfaces2-60x60"

In [12]:
# Dividimos conjunto de datos

import os
from sklearn.model_selection import RepeatedStratifiedKFold
import numpy as np

def obtener_datos(ruta_origen):
    rutas = []
    etiquetas = []
    for persona in os.listdir(ruta_origen):
        ruta_persona = os.path.join(ruta_origen, persona)
        if os.path.isdir(ruta_persona):
            imagenes = [os.path.join(ruta_persona, img) for img in os.listdir(ruta_persona)
                        if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
            rutas.extend(imagenes)
            etiquetas.extend([persona] * len(imagenes))
    return np.array(rutas), np.array(etiquetas)

def dividir_kfold_repetido(ruta_origen, k=5, repeticiones=2):
    rutas, etiquetas = obtener_datos(ruta_origen)

    rkf = RepeatedStratifiedKFold(n_splits=k, n_repeats=repeticiones, random_state=73)

    folds = []
    for i, (entrenamiento_idx, prueba_idx) in enumerate(rkf.split(rutas, etiquetas)):
        fold = {
            "entrenamiento": {
                "imagenes": rutas[entrenamiento_idx].tolist(),
                "etiquetas": etiquetas[entrenamiento_idx].tolist()
            },
            "prueba": {
                "imagenes": rutas[prueba_idx].tolist(),
                "etiquetas": etiquetas[prueba_idx].tolist()
            }
        }
        folds.append(fold)

    return folds

def obtener_datos(ruta_origen):
    rutas = []
    etiquetas = []
    for persona in os.listdir(ruta_origen):
        ruta_persona = os.path.join(ruta_origen, persona)
        if os.path.isdir(ruta_persona):
            imagenes = [os.path.join(ruta_persona, img) for img in os.listdir(ruta_persona)
                        if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
            rutas.extend(imagenes)
            etiquetas.extend([persona] * len(imagenes))
    return np.array(rutas), np.array(etiquetas)

def dividir_kfold_repetido(ruta_origen, k=5, repeticiones=2):
    rutas, etiquetas = obtener_datos(ruta_origen)

    rkf = RepeatedStratifiedKFold(n_splits=k, n_repeats=repeticiones, random_state=73)

    folds = []
    for i, (entrenamiento_idx, prueba_idx) in enumerate(rkf.split(rutas, etiquetas)):
        fold = {
            "entrenamiento": {
                "imagenes": rutas[entrenamiento_idx].tolist(),
                "etiquetas": etiquetas[entrenamiento_idx].tolist()
            },
            "prueba": {
                "imagenes": rutas[prueba_idx].tolist(),
                "etiquetas": etiquetas[prueba_idx].tolist()
            }
        }
        folds.append(fold)

    return folds


In [15]:
# En la variable "ruta_origen" se encuentran las imagenes preprocesadas para enternamiento
datos_divididos = dividir_kfold_repetido(ruta_origen)

# Selecciona el primer fold (índice 0) como ejemplo
primer_fold = datos_divididos[0]

# Accede a los datos del fold seleccionado
lista_nombres_imagenes_entrenamiento = primer_fold["entrenamiento"]["imagenes"]
lista_etiquetas_entrenamiento = primer_fold["entrenamiento"]["etiquetas"]
lista_nombres_imagenes_prueba = primer_fold["prueba"]["imagenes"]
lista_etiquetas_prueba = primer_fold["prueba"]["etiquetas"]

print("Datos divididos:", datos_divididos)
print("Primer fold:", primer_fold)
print("Lista de nombres de imágenes de entrenamiento:", lista_nombres_imagenes_entrenamiento)
print("Lista de etiquetas de entrenamiento:", lista_etiquetas_entrenamiento)
print("Lista de nombres de imágenes de prueba:", lista_nombres_imagenes_prueba)
print("Lista de etiquetas de prueba:", lista_etiquetas_prueba)

Datos divididos: [{'entrenamiento': {'imagenes': ['/content/drive/MyDrive/DMA/Eigenfaces2-60x60/Alejandro/20250321_170439_face0.jpg', '/content/drive/MyDrive/DMA/Eigenfaces2-60x60/Alejandro/IMG_1099_face0.jpg', '/content/drive/MyDrive/DMA/Eigenfaces2-60x60/Alejandro/IMG_1099_face1.jpg', '/content/drive/MyDrive/DMA/Eigenfaces2-60x60/Alejandro/1742907849763_face0.jpg', '/content/drive/MyDrive/DMA/Eigenfaces2-60x60/Alejandro/20250321_170448_face0.jpg', '/content/drive/MyDrive/DMA/Eigenfaces2-60x60/Alejandro/IMG_1100_face0.jpg', '/content/drive/MyDrive/DMA/Eigenfaces2-60x60/Alejandro/IMG_1286_face0.jpg', '/content/drive/MyDrive/DMA/Eigenfaces2-60x60/Alejandro/IMG_20250321_170457147_face0.jpg', '/content/drive/MyDrive/DMA/Eigenfaces2-60x60/Alejandro/IMG_1287_face0.jpg', '/content/drive/MyDrive/DMA/Eigenfaces2-60x60/Alejandro/IMG_1098_face0.jpg', '/content/drive/MyDrive/DMA/Eigenfaces2-60x60/Alejandro/1742907849769_face0.jpg', '/content/drive/MyDrive/DMA/Eigenfaces2-60x60/Alejandro/20250321_

# ISOMAP

In [31]:
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.manifold import Isomap
from sklearn.metrics import accuracy_score  # Or another appropriate metric
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression  # Or another classifier

def grid_search_isomap(datos_divididos, n_neighbors_range, n_components_range, k=5, repetitions=2):
    """
    Performs grid search for Isomap hyperparameters using repeated k-fold cross-validation.

    Args:
        datos_divididos (list): List of folds containing training and test data.
        n_neighbors_range (list or range): Range of values for n_neighbors.
        n_components_range (list or range): Range of values for n_components.
        k (int): Number of folds for cross-validation.
        repetitions (int): Number of repetitions for cross-validation.

    Returns:
        dict: Best hyperparameters and corresponding performance.
    """

    best_score = 0
    best_params = {}

    for n_neighbors in n_neighbors_range:
        for n_components in n_components_range:
            scores = []

            # Build the pipeline with Isomap and the classifier
            pipeline = Pipeline([
                ('isomap', Isomap(n_neighbors=n_neighbors, n_components=n_components)),
                ('classifier', LogisticRegression()) # Or another suitable classifier
            ])

            for fold in datos_divididos:  # Iterate through folds
                X_train = [np.array(Image.open(ruta).convert('L')).flatten() for ruta in fold["entrenamiento"]["imagenes"]]
                y_train = fold["entrenamiento"]["etiquetas"]
                X_test = [np.array(Image.open(ruta).convert('L')).flatten() for ruta in fold["prueba"]["imagenes"]]
                y_test = fold["prueba"]["etiquetas"]

                pipeline.fit(X_train, y_train)  # Train the pipeline
                y_pred = pipeline.predict(X_test)  # Predict on test data

                # Evaluate using appropriate metric (e.g., accuracy)
                score = accuracy_score(y_test, y_pred)
                scores.append(score)

            avg_score = np.mean(scores)  # Average score across folds

            if avg_score > best_score:
                best_score = avg_score
                best_params = {'n_neighbors': n_neighbors, 'n_components': n_components}

    return {'best_score': best_score, 'best_params': best_params}

# Example usage:
n_neighbors_range = range(5, 16)  # Adjust as needed
n_components_range = range(10, 51, 10)  # Adjust as needed

best_results = grid_search_isomap(datos_divididos, n_neighbors_range, n_components_range)

print("Best score:", best_results['best_score'])
print("Best parameters:", best_results['best_params'])

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Best score: 0.6346153846153847
Best parameters: {'n_neighbors': 14, 'n_components': 50}


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Version mejorada del codigo anterior

In [32]:
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.manifold import Isomap
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from PIL import Image
import warnings

def grid_search_isomap(datos_divididos, n_neighbors_range, n_components_range, k=5, repetitions=2):
    best_score = 0
    best_params = {}

    for n_neighbors in n_neighbors_range:
        for n_components in n_components_range:
            scores = []

            # Pipeline: escalado → reducción → clasificación
            pipeline = Pipeline([
                ('scaler', StandardScaler()),
                ('isomap', Isomap(n_neighbors=n_neighbors, n_components=n_components)),
                ('classifier', LogisticRegression(max_iter=1000, solver='lbfgs'))
            ])

            for fold in datos_divididos:
                # Cargar y vectorizar imágenes de entrenamiento
                X_train = [np.array(Image.open(ruta).convert('L')).flatten() for ruta in fold["entrenamiento"]["imagenes"]]
                y_train = fold["entrenamiento"]["etiquetas"]

                # Cargar y vectorizar imágenes de prueba
                X_test = [np.array(Image.open(ruta).convert('L')).flatten() for ruta in fold["prueba"]["imagenes"]]
                y_test = fold["prueba"]["etiquetas"]

                # Convertir a arrays numpy
                X_train = np.array(X_train)
                X_test = np.array(X_test)

                # Desactivar warning si no te interesa
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    pipeline.fit(X_train, y_train)
                    y_pred = pipeline.predict(X_test)

                score = accuracy_score(y_test, y_pred)
                scores.append(score)

            avg_score = np.mean(scores)
            if avg_score > best_score:
                best_score = avg_score
                best_params = {'n_neighbors': n_neighbors, 'n_components': n_components}

    return {'best_score': best_score, 'best_params': best_params}

# Uso ejemplo
n_neighbors_range = range(5, 16)
n_components_range = range(10, 51, 10)

best_results = grid_search_isomap(datos_divididos, n_neighbors_range, n_components_range)

print("Best score:", best_results['best_score'])
print("Best parameters:", best_results['best_params'])


Best score: 0.6211538461538463
Best parameters: {'n_neighbors': 14, 'n_components': 50}


In [16]:
def precalcular_isomap(datos_entrenamiento, n_componentes_isomap=50, n_vecinos_isomap=14):
    """
    Precalcula las ubicaciones de Isomap para las imágenes de entrenamiento.

    Args:
        datos_entrenamiento (dict): Diccionario con rutas de imágenes y etiquetas de entrenamiento.
        n_componentes_isomap (int): Número de componentes para Isomap.
        n_vecinos_isomap (int): Número de vecinos para Isomap.

    Returns:
        tuple: (ubicaciones_isomap, etiquetas_entrenamiento)
    """

    # Cargar imágenes de entrenamiento
    imagenes_entrenamiento = []
    for ruta in datos_entrenamiento["imagenes"]:
        img = Image.open(ruta).convert('L')
        imagenes_entrenamiento.append(np.array(img).flatten())
    imagenes_entrenamiento = np.array(imagenes_entrenamiento)

    # Isomap
    isomap = Isomap(n_components=n_componentes_isomap, n_neighbors=n_vecinos_isomap)
    ubicaciones_isomap = isomap.fit_transform(imagenes_entrenamiento)

    return ubicaciones_isomap, datos_entrenamiento["etiquetas"], isomap


In [18]:
# Seteamos variable para manejo mas simple, incluye etiquetas
datos_entrenamiento = datos_divididos[0]["entrenamiento"]

In [19]:
# Precalculamos datos
ubicaciones_isomap, etiquetas_entrenamiento, isomap_model = precalcular_isomap(datos_entrenamiento)

In [20]:
# Guardar datos precalculados
datos_guardar = {
    "ubicaciones_isomap": ubicaciones_isomap,
    "etiquetas_entrenamiento": etiquetas_entrenamiento,
    "n_componentes_isomap": 10,  # Guardar hiperparámetros
    "n_vecinos_isomap": 5,
    "isomap_model": isomap_model  # Store the fitted Isomap model
}

In [21]:
# Archivo parametros modelo

archivo_params = "/content/drive/MyDrive/datos_isomap.pkl"

with open(archivo_params, "wb") as f:
    pickle.dump(datos_guardar, f)

In [22]:
# Version nueva
def reconocer_cara_produccion(imagen_prueba, ruta_datos_precalculados=archivo_params):
    """
    Reconoce una cara usando datos precalculados de Isomap.

    Args:
        imagen_prueba (str): Ruta a la imagen de prueba.
        ruta_datos_precalculados (str): Ruta al archivo con los datos precalculados.

    Returns:
        list: Lista de las N etiquetas más cercanas y sus distancias.
    """

    # Cargar datos precalculados
    with open(ruta_datos_precalculados, "rb") as f:
        datos_cargados = pickle.load(f)

    ubicaciones_isomap = datos_cargados["ubicaciones_isomap"]
    etiquetas_entrenamiento = datos_cargados["etiquetas_entrenamiento"]
    # Get the pre-trained Isomap model
    isomap = datos_cargados["isomap_model"]

    # Cargar y mapear la imagen de prueba
    imagen_prueba_cargada = np.array(Image.open(imagen_prueba).convert('L')).flatten()

    # Transform the test image using the pre-trained model
    imagen_prueba_isomap = isomap.transform([imagen_prueba_cargada])

    # Calcular distancias y ranking
    distancias = euclidean_distances(imagen_prueba_isomap, ubicaciones_isomap)[0]
    indices_ordenados = np.argsort(distancias)[:3]
    ranking = [(etiquetas_entrenamiento[i], distancias[i]) for i in indices_ordenados]

    return ranking

In [24]:
# Ejemplo de uso en producción
# Access the element using its index within the list
imagen_prueba = datos_divididos[0]["prueba"]["imagenes"][50]
print(datos_divididos[0]["prueba"]["etiquetas"][50])
ranking = reconocer_cara_produccion(imagen_prueba)
print(ranking)

Guadalupe
[('Daniel', np.float64(509.1460405535991)), ('Guadalupe', np.float64(553.6282848525741)), ('Guadalupe', np.float64(553.6282848525741))]


In [25]:
def reconocer_cara_produccion2(imagen_prueba, ruta_datos_precalculados=archivo_params):
    """
    Reconoce una cara usando datos precalculados de Isomap.

    Args:
        imagen_prueba (str): Ruta a la imagen de prueba.
        ruta_datos_precalculados (str): Ruta al archivo con los datos precalculados.

    Returns:
        list: Lista de las 3 etiquetas más cercanas y distintas, con sus distancias.
    """

    # Cargar datos precalculados (same as before)
    with open(ruta_datos_precalculados, "rb") as f:
        datos_cargados = pickle.load(f)

    ubicaciones_isomap = datos_cargados["ubicaciones_isomap"]
    etiquetas_entrenamiento = datos_cargados["etiquetas_entrenamiento"]
    isomap = datos_cargados["isomap_model"]

    # Cargar y mapear la imagen de prueba (same as before)
    imagen_prueba_cargada = np.array(Image.open(imagen_prueba).convert('L')).flatten()
    imagen_prueba_isomap = isomap.transform([imagen_prueba_cargada])

    # Calcular distancias (same as before)
    distancias = euclidean_distances(imagen_prueba_isomap, ubicaciones_isomap)[0]
    indices_ordenados = np.argsort(distancias)

    # Obtener las 3 etiquetas distintas más cercanas
    ranking_01 = []
    etiquetas_encontradas = set()  # Use a set to track distinct labels
    for i in indices_ordenados:
        etiqueta = etiquetas_entrenamiento[i]
        if etiqueta not in etiquetas_encontradas:
            ranking_01.append((etiqueta, distancias[i]))
            etiquetas_encontradas.add(etiqueta)
            if len(ranking_01) == 3:
                break

    return ranking

In [26]:
print(ranking[0][0])

Daniel


In [28]:
# Ejemplo de uso en producción
# Access the element using its index within the list
#imagen_prueba = datos_divididos["prueba"]["imagenes"][67]  #Error line
imagen_prueba = datos_divididos[0]["prueba"]["imagenes"][67] # datos_divididos is a list, access by index
print(f'Etiqueta: {datos_divididos[0]["prueba"]["etiquetas"][67]}') # Access by index for etiquetas as well
ranking = reconocer_cara_produccion2(imagen_prueba)
for i in range(len(ranking)):
  print(f"Candidato {i+1}: {ranking[i][0]} - Indice: {ranking[i][1]} ")

Etiqueta: Natalia
Candidato 1: Daniel - Indice: 509.1460405535991 
Candidato 2: Guadalupe - Indice: 553.6282848525741 
Candidato 3: Guadalupe - Indice: 553.6282848525741 


In [29]:
softmax

NameError: name 'softmax' is not defined

In [ ]:
procesar_archivos_en_carpetas(ruta_entrada, ruta_salida, img_size)

para cada archivo (imagen)
  para hp1 de 1 a 100
    para hp2 de 1 a 15
      isomap(imagen)
      softmax(imagen)
      error(imagen)
      registrar_HPS_error_minimo()